# Lesson 9: Multi-Agent Systems - Emergent Behavior from Specialists

## Where We Are

Lessons 5-8 built **one** agent that handles every task itself. As tasks grow more complex, one agent juggling many tools starts to confuse itself - it forgets what it was doing, picks the wrong tool, or writes weak output because it is also worrying about research.

The fix: don't build one bigger agent. Build **several smaller specialists** and a **coordinator** that routes between them.

---

## The Mental Model

> **Think of a small consulting firm.** A client asks for a report. The **manager** does not write it - she decides who is needed next. She sends the request to the **researcher** for facts, then to the **writer** for the draft, then declares the job **done**. Each specialist does one thing well. The manager just routes.

This is the **supervisor pattern**: one coordinating node decides which worker runs next, looking at the shared message history.

---

## What We Will Build

A 3-node team that writes a short brief about a city:

| Role | Job | Tools |
|:---|:---|:---|
| **Supervisor** | Pick the next worker, or finish | none (routes) |
| **Researcher** | Gather facts about the city | `get_weather`, `get_population` |
| **Writer** | Draft a short paragraph from the facts | none (LLM only) |

All three share the same message history (the State). The supervisor reads that history and outputs a **structured decision** - `researcher`, `writer`, or `FINISH`.

---

## Graph Structure

```
                       START
                         |
                         v
                  +-------------+
          +------>| supervisor  |--FINISH-->END
          |       +-------------+
          |          /        \
          |   researcher    writer
          |     /              \
          +----+                +----+
```

Every worker hands control back to the supervisor. The supervisor decides the next move. The loop ends when the supervisor returns `FINISH`.

---

## Why "Emergent"?

We never wrote `researcher -> writer` as an edge. We never wrote `if all facts gathered, then write`. The supervisor figures that out from the conversation. The execution order emerges from the LLM's reading of the room - not from hard-coded logic.

---

## Step 1: Setup

In [ ]:
%pip install -q langgraph langchain langchain-openai python-dotenv

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set"
print("API key loaded.")

---

## Step 2: Tools (for the Researcher only)

Same fake lookups from earlier lessons. Only the researcher will see these.

In [ ]:
from langchain_core.tools import tool

WEATHER_DATA = {
    "new york": "72F, Sunny",
    "london": "58F, Cloudy",
    "tokyo": "68F, Rainy",
}
POPULATION_DATA = {
    "new york": "8.3 million",
    "london": "8.9 million",
    "tokyo": "13.9 million",
}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return WEATHER_DATA.get(city.lower(), f"No weather data for {city}")

@tool
def get_population(city: str) -> str:
    """Get population of a city."""
    return POPULATION_DATA.get(city.lower(), f"No population data for {city}")

researcher_tools = [get_weather, get_population]

---

## Step 3: Shared State

All three workers read and write the same `messages` list. That is the **shared scratchpad** - the team's conversation.

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

---

## Step 4: The Supervisor

The supervisor's only job is **routing**. To make its decision reliable we use **structured output** - the LLM must return one of three exact strings: `researcher`, `writer`, or `FINISH`.

No structured output = the LLM might say *"I think the researcher should go next, but maybe..."* and we'd be stuck parsing prose. Structured output forces a clean choice.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

WORKERS = ["researcher", "writer"]
OPTIONS = WORKERS + ["FINISH"]

class Route(BaseModel):
    """Decide which worker runs next, or finish."""
    next: Literal["researcher", "writer", "FINISH"] = Field(
        description="Who should act next. Use FINISH only when the user's request is fully answered."
    )

supervisor_prompt = (
    "You are a supervisor managing a team of workers: researcher, writer. "
    "Given the conversation so far, decide who should act next. "
    "The researcher looks up factual data using tools. "
    "The writer drafts a short paragraph using facts already in the conversation. "
    "Respond with FINISH when the user's original request has been fully delivered."
)

supervisor_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Route)

def supervisor(state: State) -> dict:
    messages = [SystemMessage(content=supervisor_prompt)] + state["messages"]
    decision = supervisor_llm.invoke(messages)
    print(f"  [supervisor] -> {decision.next}")
    return {"next": decision.next}

Notice `supervisor` returns `{"next": ...}` - but our State only has `messages`. We need to add a `next` field to State so the routing decision is preserved between calls.

In [ ]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next: str

---

## Step 5: The Researcher

A standard tool-using agent (Lesson 5 pattern), but scoped to a tight prompt: gather facts, stop.

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

researcher_prompt = (
    "You are a researcher. Use the tools to gather the facts requested. "
    "Report the facts plainly. Do not write a paragraph. Do not editorialize."
)
researcher_agent = create_react_agent(
    ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=researcher_tools,
    prompt=researcher_prompt,
)

def researcher(state: State) -> dict:
    result = researcher_agent.invoke({"messages": state["messages"]})
    last = result["messages"][-1]
    print(f"  [researcher] {last.content[:120]}")
    return {"messages": [HumanMessage(content=last.content, name="researcher")]}

**Why wrap the result in a `HumanMessage` named `researcher`?** The researcher's internal tool calls and AI responses are private to its own loop. The team only needs the *final answer* on the shared scratchpad. Tagging it with a `name` lets the supervisor and writer see *who said what*.

---

## Step 6: The Writer

No tools. Reads whatever facts are in the conversation, drafts one short paragraph.

In [ ]:
writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)
writer_prompt = (
    "You are a writer. Using only the facts already in the conversation, "
    "draft a single short paragraph (2-3 sentences) that answers the user's original request. "
    "Do not invent facts. Do not ask follow-up questions."
)

def writer(state: State) -> dict:
    messages = [SystemMessage(content=writer_prompt)] + state["messages"]
    result = writer_llm.invoke(messages)
    print(f"  [writer] {result.content[:120]}")
    return {"messages": [HumanMessage(content=result.content, name="writer")]}

---

## Step 7: Wire the Graph

Workers always hand control back to the supervisor. The supervisor uses its `next` field to route via a conditional edge.

In [ ]:
from langgraph.graph import StateGraph, START, END

def route_from_supervisor(state: State) -> str:
    decision = state["next"]
    if decision == "FINISH":
        return END
    return decision

workflow = StateGraph(State)
workflow.add_node("supervisor", supervisor)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)

workflow.add_edge(START, "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {"researcher": "researcher", "writer": "writer", END: END},
)
workflow.add_edge("researcher", "supervisor")
workflow.add_edge("writer", "supervisor")

app = workflow.compile()
print("Multi-agent graph compiled.")

---

## Step 8: Visualize the Graph

Notice the **hub-and-spoke** shape. Every worker connects to the supervisor, never directly to another worker.

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

---

## Step 9: Run the Team

Ask for a short brief about Tokyo. Watch the trace - the supervisor decides every hop.

In [ ]:
result = app.invoke({
    "messages": [("user", "Write a short brief about Tokyo's current weather and population.")],
})

print("\n=== FINAL ANSWER ===")
print(result["messages"][-1].content)

You should see something like:

```
  [supervisor] -> researcher
  [researcher] Tokyo - Weather: 68F, Rainy. Population: 13.9 million.
  [supervisor] -> writer
  [writer] Tokyo is currently 68F and rainy, with a population of about 13.9 million...
  [supervisor] -> FINISH
```

We never wrote `if researcher_done then writer`. The supervisor inferred it.

---

## Step 10: A Harder Request

Try a request that needs the team to act differently. What if the user just asks for facts, no paragraph?

In [ ]:
result2 = app.invoke({
    "messages": [("user", "Just give me the raw weather data for London. Don't write anything fancy.")],
})

print("\n=== FINAL ANSWER ===")
print(result2["messages"][-1].content)

If the supervisor is doing its job, you'll see it skip the writer entirely - go to the researcher, then FINISH. **Same graph, different execution path**, driven entirely by reading the user's intent.

---

## Summary

### What Changed from Lesson 8

| Lessons 5-8 (Single Agent) | Lesson 9 (Multi-Agent) |
|:---|:---|
| One LLM does everything | Several LLMs, each with a narrow job |
| Routing is hard-coded in conditional edges | Routing decided by an LLM (the supervisor) |
| Tools live on one agent | Tools live only where they're needed |
| Execution path is fixed by the graph | Execution path emerges from the conversation |

---

### Anatomy of the Supervisor Pattern

```
        START --> supervisor --(routes)--> worker_N --> supervisor --> ... --> END
                       ^                                    |
                       +------------------------------------+
```

Every worker returns to the supervisor. The supervisor's structured-output decision drives the next hop. The loop ends when it returns `FINISH`.

---

### API Reference

| Component | Purpose |
|:---|:---|
| `with_structured_output(Schema)` | Forces the LLM to return a typed object, not prose |
| `Literal[...]` in the Pydantic schema | Constrains the LLM's choice to a closed set |
| `HumanMessage(content=..., name="X")` | Tags a message with the worker that produced it |
| `create_react_agent(llm, tools, prompt)` | Prebuilt one-shot agent that does the agent/tool loop |
| Shared `messages` State | The team's scratchpad - everyone reads, everyone writes |

---

### Key Takeaways

> 1. One big agent struggles with breadth - split into specialists with narrow prompts
> 2. A **supervisor** node decides the next hop using **structured output**
> 3. Workers always hand control back to the supervisor (hub and spoke)
> 4. The execution path is **emergent** - not pre-wired - because routing is LLM-driven
> 5. Tag worker outputs with `name=...` so the team can tell who said what
> 6. Hide each worker's private chatter - only expose its final answer on the shared scratchpad

---

**Next Lesson**: We have built increasingly complex graphs and have been printing trace lines to understand what's happening. Lesson 10 introduces **LangGraph Studio** - a visual debugger that shows the graph, lets you step through executions, and inspect State at every checkpoint without writing a single print statement.